In [1]:
import requests
import urllib.robotparser
from urllib.parse import urljoin

# ==============================================================================
# CLASES DE ESTRUCTURA DE DATOS
# ==============================================================================

class FuenteWeb:
    def __init__(self, nombre: str, base_url: str, rutas_objetivo: list = None, api_recomendada: bool = False, descripcion_api: str = ""):
        self.nombre: str = nombre
        self.base_url: str = base_url
        self.rutas_objetivo: list = rutas_objetivo if rutas_objetivo is not None else []
        self.api_recomendada: bool = api_recomendada
        self.descripcion_api: str = descripcion_api if descripcion_api else "No requiere API obligatoria."

class ResultadoRuta:
    def __init__(self, ruta: str, url: str, permitido: bool):
        self.ruta: str = ruta
        self.url: str = url
        self.permitido: bool = permitido

class ResultadoAuditoria:
    def __init__(self, fuente: FuenteWeb, robots_url: str, status_code: int = None, 
                 existe_robots: bool = False, rutas: list = None, sitemaps: list = None, 
                 crawl_delays: list = None, error: str = None, nivel_acceso: str = "INCIERTO", mensaje: str = ""):
        
        self.fuente: FuenteWeb = fuente
        self.robots_url: str = robots_url
        self.status_code: int = status_code
        self.existe_robots: bool = existe_robots
        self.rutas: list = rutas if rutas is not None else []
        self.sitemaps: list = sitemaps if sitemaps is not None else []
        self.crawl_delays: list = crawl_delays if crawl_delays is not None else []
        self.nivel_acceso: str = nivel_acceso
        self.mensaje: str = mensaje
        self.error: str = error

# ==============================================================================
# CLASE PRINCIPAL DE AUDITORÍA
# ==============================================================================

class AuditorRobots:
    def __init__(self, user_agent="ProyectoFakeReviews/1.0"):
        self.user_agent = user_agent

    def obtener_robots_url(self, base_url):
        return urljoin(base_url, "/robots.txt")

    def descargar_robots(self, robots_url):
        try:
            response = requests.get(
                robots_url,
                headers={"User-Agent": self.user_agent},
                timeout=10
            )
            return response.status_code, response.text, None
        except requests.RequestException as e:
            return None, "", str(e)

    def extraer_sitemaps(self, contenido_robots):
        sitemaps = []
        for linea in contenido_robots.splitlines():
            linea_limpia = linea.strip()
            if linea_limpia.lower().startswith("sitemap:"):
                sitemap = linea_limpia.split(":", 1)[1].strip()
                sitemaps.append(sitemap)
        return sitemaps

    def extraer_crawl_delay(self, contenido_robots):
        delays = []
        for linea in contenido_robots.splitlines():
            linea_limpia = linea.strip().lower()
            if linea_limpia.startswith("crawl-delay"):
                delays.append(linea_limpia)
        return delays

    def evaluar_rutas(self, fuente, robots_url):
        parser = urllib.robotparser.RobotFileParser()
        parser.set_url(robots_url)
        try:
            parser.read()
        except Exception:
            return []
            
        resultados = []
        for ruta in fuente.rutas_objetivo:
            url_completa = urljoin(fuente.base_url, ruta)
            permitido = parser.can_fetch(self.user_agent, url_completa)
            resultados.append(
                ResultadoRuta(
                    ruta=ruta,
                    url=url_completa,
                    permitido=permitido
                )
            )
        return resultados

    def clasificar_acceso(self, fuente, rutas, existe_robots):
        if not existe_robots:
            return (
                "INCIERTO",
                "Acceso incierto: requiere revisión manual de robots.txt y términos de uso."
            )
            
        if len(rutas) == 0:
            return (
                "INCIERTO",
                "Acceso incierto: no se pudieron evaluar las rutas objetivo."
            )
            
        total = len(rutas)
        permitidas = sum(1 for ruta in rutas if ruta.permitido) 
        
        if permitidas == total and not fuente.api_recomendada:
            return (
                "ALTO",
                "Apto para scraping no intrusivo con requests y BeautifulSoup."
            )
        elif permitidas == total and fuente.api_recomendada:
            return (
                "API_RECOMENDADA",
                "Aunque las rutas son accesibles, se recomienda utilizar la API oficial para extraer las reseñas."
            )
        elif 0 < permitidas < total:
            return (
                "MEDIO",
                "Acceso parcial: priorizar la API oficial o realizar scraping limitado con pausas largas."
            )
        else:
            return (
                "RESTRINGIDO",
                "No apto para scraping directo: utilizar los endpoints de la API sujeta a los términos de la plataforma."
            )

    def auditar(self, fuente):
        robots_url = self.obtener_robots_url(fuente.base_url)
        status_code, contenido, error = self.descargar_robots(robots_url)
        
        if error:
            return ResultadoAuditoria(
                fuente=fuente,
                robots_url=robots_url,
                status_code=status_code,
                existe_robots=False,
                nivel_acceso="INCIERTO",
                mensaje="No se pudo consultar robots.txt. Revisar manualmente.",
                error=error
            )
            
        existe_robots = status_code == 200
        
        if not existe_robots:
            return ResultadoAuditoria(
                fuente=fuente,
                robots_url=robots_url,
                status_code=status_code,
                existe_robots=False,
                nivel_acceso="INCIERTO",
                mensaje="robots.txt no disponible o no accesible. Revisar términos de uso."
            )
            
        sitemaps = self.extraer_sitemaps(contenido)
        crawl_delays = self.extraer_crawl_delay(contenido)
        rutas = self.evaluar_rutas(fuente, robots_url)
        
        nivel, mensaje = self.clasificar_acceso(
            fuente=fuente,
            rutas=rutas,
            existe_robots=existe_robots
        )
        
        return ResultadoAuditoria(
            fuente=fuente,
            robots_url=robots_url,
            status_code=status_code,
            existe_robots=existe_robots,
            rutas=rutas,
            sitemaps=sitemaps,
            crawl_delays=crawl_delays,
            nivel_acceso=nivel,
            mensaje=mensaje
        )

# ==============================================================================
# CLASE DE REPORTE
# ==============================================================================

class ReporteAuditoria:
    def imprimir(self, resultado):
        print("=" * 75)
        print(f"Fuente: {resultado.fuente.nombre}")
        print(f"URL base: {resultado.fuente.base_url}")
        print(f"robots.txt: {resultado.robots_url}")
        print(f"Estado HTTP: {resultado.status_code}")
        
        if resultado.error:
            print(f"Error: {resultado.error}")
            
        print(f"\nNivel de acceso estimado: {resultado.nivel_acceso}")
        print(f"Recomendación: {resultado.mensaje}")
        print(f"Uso de API: {resultado.fuente.descripcion_api}")
        
        print("\nRutas evaluadas:")
        if resultado.rutas:
            for ruta in resultado.rutas:
                estado = "PERMITIDA" if ruta.permitido else "NO PERMITIDA"
                print(f"- {ruta.ruta}: {estado}")
        else:
            print("- No se evaluaron rutas.")
            
        print("\nSitemaps encontrados:")
        if resultado.sitemaps:
            for sitemap in resultado.sitemaps[:5]:
                print(f"- {sitemap}")
        else:
            print("- No se encontraron sitemaps declarados.")
            
        print("\nCrawl-delay:")
        if resultado.crawl_delays:
            for delay in resultado.crawl_delays:
                print(f"- {delay}")
        else:
            print("- No se encontró crawl-delay explícito.")
        print("\n")

# ==============================================================================
# EJECUCIÓN DEL ANÁLISIS
# ==============================================================================

def main():
    fuentes = [
        FuenteWeb(
            nombre="Steam Store",
            base_url="https://store.steampowered.com",
            rutas_objetivo=[
                "/app/",           # Páginas de videojuegos
                "/appreviews/",    # Endpoints directos de reseñas
                "/search/"         # Búsqueda de catálogo
            ],
            api_recomendada=True,
            descripcion_api="Altamente recomendada. Steamworks Web API y Storefront API facilitan la extracción masiva de reseñas y metadatos de usuarios."
        ),
        FuenteWeb(
            nombre="OpenCritic",
            base_url="https://opencritic.com",
            rutas_objetivo=[
                "/game/",          # Páginas de los juegos y reseñas de críticos
                "/api/"            # Endpoints internos del sitio
            ],
            api_recomendada=True,
            descripcion_api="Recomendada. OpenCritic posee una API, aunque las políticas de uso comercial o masivo pueden requerir una key de acceso."
        )
    ]
    
    auditor = AuditorRobots(user_agent="ProyectoFakeReviews/1.0")
    reporte = ReporteAuditoria() 
    
    for fuente in fuentes:
        resultado = auditor.auditar(fuente)
        reporte.imprimir(resultado)

if __name__ == "__main__":
    main()

Fuente: Steam Store
URL base: https://store.steampowered.com
robots.txt: https://store.steampowered.com/robots.txt
Estado HTTP: 200

Nivel de acceso estimado: API_RECOMENDADA
Recomendación: Aunque las rutas son accesibles, se recomienda utilizar la API oficial para extraer las reseñas.
Uso de API: Altamente recomendada. Steamworks Web API y Storefront API facilitan la extracción masiva de reseñas y metadatos de usuarios.

Rutas evaluadas:
- /app/: PERMITIDA
- /appreviews/: PERMITIDA
- /search/: PERMITIDA

Sitemaps encontrados:
- No se encontraron sitemaps declarados.

Crawl-delay:
- No se encontró crawl-delay explícito.


Fuente: OpenCritic
URL base: https://opencritic.com
robots.txt: https://opencritic.com/robots.txt
Estado HTTP: 200

Nivel de acceso estimado: API_RECOMENDADA
Recomendación: Aunque las rutas son accesibles, se recomienda utilizar la API oficial para extraer las reseñas.
Uso de API: Recomendada. OpenCritic posee una API, aunque las políticas de uso comercial o masivo pu